In [2]:
import pandas as pd
import os
from tqdm.auto import tqdm
import platform
import sqlite3

pd.options.mode.chained_assignment = None  # default='warn'


In [3]:
#For each example in the blimp dataset, find the length and save 
#For examples with the two sentences having different lengths, decide what to do (easier is first to just save both the lengths separately, can always be combined later)

if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'#Given a set of details, return a dataframe compiled with the details

RESULTS_ROOT = os.path.join(ROOT, "results")
SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)


#Other Utils
import io
import sys
import contextlib

@contextlib.contextmanager
def silence_prints():
    sys.stdout, old = io.StringIO(), sys.stdout
    try:
        yield
    finally:
        sys.stdout = old


In [4]:
blimp_root = "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/evaluation_pipeline/filter-data/blimp_filtered"
blimp_supplement_root = "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/evaluation_pipeline/filter-data/supplement_filtered"

blimp_files_list = sorted(os.listdir(blimp_root))
blimp_supplement_files_list = sorted(os.listdir(blimp_supplement_root))
blimp_files_list

['anaphor_agreement.json',
 'argument_structure.json',
 'binding.json',
 'control_raising.json',
 'determiner_noun_agreement.json',
 'ellipsis.json',
 'filler_gap.json',
 'irregular_forms.json',
 'island_effects.json',
 'npi_licensing.json',
 'quantifiers.json',
 'subject_verb_agreement.json']

In [5]:
#To Do - For each Model Name in the BlimpResultsSubtaskwiseBreakdown table, manage to link it to corresponding RowIndex from BlimpSubtaskGroundTruth table

ground_truth_df = pd.read_sql_query("SELECT RowIdx, SentenceGood, SentenceBad, Subtask FROM BlimpSubtaskGroundTruth", conn)

model_names_list = c.execute("SELECT DISTINCT ModelName FROM BlimpResultsSubtaskwiseBreakdown").fetchall()
model_names_list = [x[0] for x in model_names_list]

model_names_list

['out-babylm_full_bpe-2x2-nomask-1709512418',
 'out-babylm_full_bpe-4x4-nomask-1709506109',
 'out-babylm_full_bpe-4x4-nomask-5444724',
 'out-babylm_full_bpe-6x6-mask_e002-5757736',
 'out-babylm_full_bpe-6x6-mask_e100-5757737',
 'out-babylm_full_bpe-6x6-nomask-5492134',
 'out-babylm_full_bpe-8x8-nomask-5492054',
 'out-babylm_full_bpe_100M_8k-12x12-mask_ee002_em01-7761454_s1337',
 'out-babylm_full_bpe_100M_8k-12x12-mask_ee002_em01-7920299',
 'out-babylm_full_bpe_100M_8k-12x12-mask_ee002_em05-8058703',
 'out-babylm_full_bpe_100M_8k-12x12-mask_ee002_em10-8058704',
 'out-babylm_full_bpe_100M_8k-12x12-nomask-7761451_s1337',
 'out-babylm_full_bpe_100M_8k-12x12-nomask-7915350_s1337',
 'out-babylm_full_bpe_100M_8k-12x12-nomask-7938712',
 'out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8111938',
 'out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8111939_s42',
 'out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8111940_s2347',
 'out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8111941_s616',
 'out-b

In [6]:
ground_truth_df

,RowIdx,SentenceGood,SentenceBad,Subtask
0,1,Katherine can't help herself.,Katherine can't help himself.,anaphor_agreement
1,2,Karla could listen to herself.,Karla could listen to himself.,anaphor_agreement
2,3,Marie won't think about herself.,Marie won't think about itself.,anaphor_agreement
3,4,Mark hasn't discussed himself.,Mark hasn't discussed itself.,anaphor_agreement
4,5,Stephen impressed himself.,Stephen impressed itself.,anaphor_agreement
...,...,...,...,...
63275,63276,"David: Did we say that you left?\nSarah: No, y...","David: Did we say that you left?\nSarah: No, t...",turn_taking
63276,63277,"A: Did we say that you finished?\nB: No, you d...","A: Did we say that you finished?\nB: No, they ...",turn_taking
63277,63278,A: Did we say that we will find something to d...,A: Did we say that we will find something to d...,turn_taking
63278,63279,A: Should we say that she will consider asking...,A: Should we say that she will consider asking...,turn_taking


In [14]:
model_subtask_results = pd.read_sql_query("SELECT * FROM BlimpResultsSubtaskwiseBreakdown WHERE ModelName = '{}'".format(model_names_list[0]), conn)
model_subtask_results

,RowIndex,PredictedSentence,ModelName,SubtaskName
0,0,A lot of patients who can sell some couch didn...,out-babylm_full_bpe-2x2-nomask-1709512418,binding
1,1,A lot of actresses that thought about Alice he...,out-babylm_full_bpe-2x2-nomask-1709512418,binding
2,2,A guy that has seen the wheelbarrow notices hi...,out-babylm_full_bpe-2x2-nomask-1709512418,binding
3,3,The children that were complaining about Grego...,out-babylm_full_bpe-2x2-nomask-1709512418,binding
4,4,Every boy that isn't selling the grocery store...,out-babylm_full_bpe-2x2-nomask-1709512418,binding
...,...,...,...,...
63275,1960,All teachers drunk.,out-babylm_full_bpe-2x2-nomask-1709512418,irregular_forms
63276,1961,Every cat broke that skateboard.,out-babylm_full_bpe-2x2-nomask-1709512418,irregular_forms
63277,1962,Every dancer worn that sweater.,out-babylm_full_bpe-2x2-nomask-1709512418,irregular_forms
63278,1963,Ann gone to some river.,out-babylm_full_bpe-2x2-nomask-1709512418,irregular_forms


In [ ]:
def combine_gt_subtask_results(gt_df, model_subtask_df):
    """
    Given a ground truth dataframe and a subtask results dataframe, return a combined dataframe with the relevant columns

    Args:
        gt_df (pd.DataFrame): Ground truth dataframe
        subtask_df (pd.DataFrame): Subtask results dataframe

    Returns:
        pd.DataFrame: Combined dataframe
    """

    assert gt_df.shape[0] == model_subtask_df.shape[0], "Ground truth and subtask results dataframes should have the same number of rows"
    #print("Shape of gt_df: ", gt_df.shape)
    #print("Shape of model_subtask_df: ", model_subtask_df.shape)
    #Join SubtaskName and SentenceGood first  and then on SubtaskName and SentenceBad

    # combined_df = model_subtask_df.copy(deep=True)
    # combined_df["SentenceGood"] = gt_df["SentenceGood"]
    # combined_df["SentenceBad"] = gt_df["SentenceBad"]
    # combined_df["Subtask"] = gt_df["Subtask"]
    # combined_df["RowIndex"] = gt_df["RowIndex"]

    for subtask in model_subtask_df["SubtaskName"].unique():
        subtask_gt = gt_df[gt_df["Subtask"] == subtask]
        subtask_model = model_subtask_df[model_subtask_df["SubtaskName"] == subtask]
        
        subtask_model["SentenceGood"] = subtask_gt["SentenceGood"].values
        subtask_model["SentenceBad"] = subtask_gt["SentenceBad"].values
        subtask_model["Subtask"] = subtask_gt["Subtask"].values
        subtask_model["RowIdx"] = subtask_gt["RowIdx"].values

        if subtask == model_subtask_df["SubtaskName"].unique()[0]:
            combined_df = subtask_model
        else:
            combined_df = pd.concat([combined_df, subtask_model], axis=0)

    def check_strings(predicted_sentence, sentence_good, sentence_bad):
        if predicted_sentence.replace("\\n", "\n") == sentence_good.replace("\\n", "\n"):
            return "Good"
        elif predicted_sentence.replace("\\n", "\n") == sentence_bad.replace("\\n", "\n"):
            return "Bad"
        elif predicted_sentence.replace('"', "").replace("\\n", "\n") == sentence_good.replace('"', "").replace("\\n", "\n"):
            return "Good"
        elif predicted_sentence.replace('"', "").replace("\\n", "\n") == sentence_bad.replace('"', "").replace("\\n", "\n"):
            return "Bad"
        else:
            return None

    #Check that model's PredictedSentence is the same as the SentenceGood or SentenceBad
    # combined_df["PredictedClass"] = combined_df.apply(lambda x: "Good" if x["PredictedSentence"].apply(repr) == x["SentenceGood"].apply(repr) else ("Bad" if x["PredictedSentence"].apply(repr) == x["SentenceBad"].apply(repr) else None), axis=1)
    combined_df["PredictedClass"] = combined_df.apply(lambda x: check_strings(x["PredictedSentence"], x["SentenceGood"], x["SentenceBad"]), axis=1)

    assert all(combined_df["PredictedClass"].notnull()), "PredictedClass should not be null"
    #assert all(combined_df["PredictedClass"].isin(["Good", "Bad"])), "PredictedClass should be either Good or Bad"
    assert all(combined_df["Subtask"] == combined_df["SubtaskName"]), "Subtask and SubtaskName should be the same"

    #Write combined df back to Database but only RowIndex and PredictedClass
    
    combined_write_df = combined_df[["RowIndex", "PredictedSentence", "ModelName", "SubtaskName", "PredictedClass", "RowIdx"]]
    
    # for idx, row in tqdm(combined_write_df.iterrows()):
    #     c.execute("UPDATE BlimpResultsSubtaskwiseBreakdown SET PredictedClass = ?, RowIdx = ? WHERE RowIndex = ? AND ModelName = ? AND SubtaskName = ? AND PredictedSentence = ?", (row["PredictedClass"], row["RowIdx"], row["RowIndex"], row["ModelName"], row["SubtaskName"], row["PredictedSentence"]))

    # update_query = "UPDATE BlimpResultsSubtaskwiseBreakdown SET PredictedClass = ?, RowIdx = ? WHERE RowIndex = ? AND ModelName = ? AND SubtaskName = ? AND PredictedSentence = ?"

    # update_data = [(row["PredictedClass"], row["RowIdx"], row["RowIndex"], row["ModelName"], row["SubtaskName"], row["PredictedSentence"]) for idx, row in combined_write_df.iterrows()]

    # c.executemany(update_query, update_data)
    # conn.commit()
    combined_write_df.to_sql("BlimpResultsSubtaskwiseBreakdownTemp", conn, if_exists="append", index=False)
    conn.commit()

    return combined_df

# for i, model_name in enumerate(tqdm(model_names_list)):
#     model_subtask_results = pd.read_sql_query("SELECT * FROM BlimpResultsSubtaskwiseBreakdown WHERE ModelName = '{}'".format(model_name), conn)
#     a1 = combine_gt_subtask_results(ground_truth_df, model_subtask_results)
    # except Exception as e:
    #     print("Error for model_name: ", model_name)
    #     print(e)
    #     break


  0%|          | 0/216 [00:00<?, ?it/s]

In [25]:
results_df = pd.read_sql_query("""WITH GoodPercentage AS (SELECT 
    ModelName, 
    SubtaskName, 
    ROUND(100.0 * SUM(CASE WHEN PredictedClass = 'Good' THEN 1 ELSE 0 END) / COUNT(*), 2) AS Good_Percentage
FROM BlimpResultsSubtaskwiseBreakdown
GROUP BY ModelName, SubtaskName),
                               
GoodPercentageWithAvg AS (
SELECT
    Subtask, AVG(SentenceGoodWordLength) AS AvgSentenceGoodWordLength, AVG(SentenceBadWordLength) AS AvgSentenceBadWordLength, AVG(SentenceGoodCharacterLength) AS AvgSentenceGoodCharLength, AVG(SentenceBadCharacterLength) AS AvgSentenceBadCharLength
FROM BlimpSubtaskGroundTruth
GROUP BY Subtask 
)

SELECT GoodPercentage.ModelName, Model.ModelID, GoodPercentage.SubtaskName,  GoodPercentage.Good_Percentage, GoodPercentageWithAvg.AvgSentenceGoodWordLength, GoodPercentageWithAvg.AvgSentenceBadWordLength, GoodPercentageWithAvg.AvgSentenceGoodCharLength, GoodPercentageWithAvg.AvgSentenceBadCharLength
FROM GoodPercentage
Join Model ON 
Model.OutputFolderName=GoodPercentage.ModelName
JOIN GoodPercentageWithAvg ON GoodPercentage.SubtaskName = GoodPercentageWithAvg.Subtask
                               

""", conn)

#results_df = results_df.pivot(index=["ModelName", "ModelID"], columns="SubtaskName", values="Good_Percentage")

results_df

,ModelName,ModelID,SubtaskName,Good_Percentage,AvgSentenceGoodWordLength,AvgSentenceBadWordLength,AvgSentenceGoodCharLength,AvgSentenceBadCharLength
0,out-babylm_full_bpe-4x4-nomask-5444724,5444724,anaphor_agreement,45.30,4.149796,4.149796,30.089468,29.853783
1,out-babylm_full_bpe-4x4-nomask-5444724,5444724,argument_structure,56.05,4.855601,5.049830,31.239573,32.153007
2,out-babylm_full_bpe-4x4-nomask-5444724,5444724,binding,56.19,7.478332,7.478332,49.586227,50.560255
3,out-babylm_full_bpe-4x4-nomask-5444724,5444724,control_raising,53.16,9.731772,9.716748,56.511047,56.407645
4,out-babylm_full_bpe-4x4-nomask-5444724,5444724,determiner_noun_agreement,50.11,5.773270,5.773270,37.386370,37.407452
...,...,...,...,...,...,...,...,...
3531,out-babylm_wocdes_full_bpe-4x4-nomask-5445338,5445338,qa_congruence_tricky,41.82,7.739394,7.090909,40.254545,36.787879
3532,out-babylm_wocdes_full_bpe-4x4-nomask-5445338,5445338,quantifiers,43.59,7.393869,7.393869,44.700155,44.238794
3533,out-babylm_wocdes_full_bpe-4x4-nomask-5445338,5445338,subject_aux_inversion,47.91,13.884850,14.105148,79.792144,80.453037
3534,out-babylm_wocdes_full_bpe-4x4-nomask-5445338,5445338,subject_verb_agreement,47.88,6.189883,6.189883,38.897561,38.935321


In [18]:
gt_len_df = pd.read_sql_query(
    """
    SELECT RowIdx, SentenceGood, SentenceBad FROM BlimpSubtaskGroundTruth
    """, conn)



In [20]:
gt_len_df["SentenceGoodCharacterLength"] = gt_len_df["SentenceGood"].apply(len)
gt_len_df["SentenceBadCharacterLength"] = gt_len_df["SentenceBad"].apply(len)
gt_len_df["SentenceGoodWordLength"] = gt_len_df["SentenceGood"].apply(lambda x: len(x.split()))
gt_len_df["SentenceBadWordLength"] = gt_len_df["SentenceBad"].apply(lambda x: len(x.split()))

gt_len_df

,RowIdx,SentenceGood,SentenceBad,SentenceGoodCharacterLength,SentenceBadCharacterLength,SentenceGoodWordLength,SentenceBadWordLength
0,1,Katherine can't help herself.,Katherine can't help himself.,29,29,4,4
1,2,Karla could listen to herself.,Karla could listen to himself.,30,30,5,5
2,3,Marie won't think about herself.,Marie won't think about itself.,32,31,5,5
3,4,Mark hasn't discussed himself.,Mark hasn't discussed itself.,30,29,4,4
4,5,Stephen impressed himself.,Stephen impressed itself.,26,25,3,3
...,...,...,...,...,...,...,...
63275,63276,"David: Did we say that you left?\nSarah: No, y...","David: Did we say that you left?\nSarah: No, t...",64,65,13,13
63276,63277,"A: Did we say that you finished?\nB: No, you d...","A: Did we say that you finished?\nB: No, they ...",51,52,11,11
63277,63278,A: Did we say that we will find something to d...,A: Did we say that we will find something to d...,73,74,16,16
63278,63279,A: Should we say that she will consider asking...,A: Should we say that she will consider asking...,86,87,16,16


In [ ]:
# for idx, row in tqdm(gt_len_df.iterrows()):
#     c.execute("UPDATE BlimpSubtaskGroundTruth SET SentenceGoodCharacterLength = ?, SentenceBadCharacterLength = ?, SentenceGoodWordLength = ?, SentenceBadWordLength = ? WHERE RowIdx = ?", (row["SentenceGoodCharacterLength"], row["SentenceBadCharacterLength"], row["SentenceGoodWordLength"], row["SentenceBadWordLength"], row["RowIdx"]))
# conn.commit()

0it [00:00, ?it/s]

In [17]:
#Function Block to get GroundTruth for Blimp

# blimp_masterlist_raw_df = []
# for filename in blimp_files_list:
#     blimp_subtasK_df = pd.read_json(os.path.join(blimp_root, filename), lines=True)
#     blimp_subtasK_df["subtask_filename"] = filename
#     blimp_subtasK_df["subtask"] = filename.split(".")[0]
#     blimp_subtasK_df["task_type"] = "blimp"
#     blimp_masterlist_raw_df.append(blimp_subtasK_df)

# for filename in blimp_supplement_files_list:
#     blimp_subtasK_df = pd.read_json(os.path.join(blimp_supplement_root, filename), lines=True)
#     blimp_subtasK_df["subtask_filename"] = filename
#     blimp_subtasK_df["subtask"] = filename.split(".")[0]
#     blimp_subtasK_df["task_type"] = "blimp_supplement"
#     blimp_masterlist_raw_df.append(blimp_subtasK_df)

# blimp_masterlist_raw_df = pd.concat(blimp_masterlist_raw_df)
# blimp_masterlist_raw_df["row_index"] = blimp_masterlist_raw_df.index
# blimp_masterlist_df = blimp_masterlist_raw_df[["row_index", "sentence_good", "sentence_bad", "subtask_filename", "subtask", "task_type"]]
# blimp_masterlist_df

In [11]:
# blimp_masterlist_df.to_sql("BlimpSubtaskGroundTruth", conn, if_exists="replace", index=False)

63280

In [15]:
# #Function block to read all predicted sentences for all models and store them in the database

# #Output Dump Folders Path
# import json
# output_dump_folder_root = "/media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump"
# #subtasks_predictions_df = []
# test_subtask_list = []
# for model_name in tqdm(os.listdir(output_dump_folder_root)):
#     if not os.path.isdir(os.path.join(output_dump_folder_root, model_name)):
#         continue
#     if "zeroshot" in os.listdir(os.path.join(output_dump_folder_root, model_name)):
#         model_output_dump_folder = os.path.join(output_dump_folder_root, model_name, "zeroshot")
#         if len(os.listdir(model_output_dump_folder)) == 17:
            
#             for subtask_name in os.listdir(model_output_dump_folder):
#                 #predictions_file = os.path.join(model_output_dump_folder, subtask_name, "predictions.txt")
#                 score_file = os.path.join(model_output_dump_folder, subtask_name, "eval_results.json")
#                 try:
#                     with open(score_file, "r") as f:
#                         score = json.load(f)["eval_accuracy"]
#                         test_subtask_list.append(
#                             {
#                                 "model_name": model_name,
#                                 "subtask": subtask_name,
#                                 "score": score
#                             }
#                         )
#                     # subtask_predictions_df = pd.read_csv(predictions_file, sep="\t")
#                     # subtask_predictions_df["model_name"] = model_name
#                     # subtask_predictions_df["subtask"] = subtask_name
#                     # #subtasks_predictions_df.append(subtask_predictions_df)
#                     # subtask_predictions_df = subtask_predictions_df.rename(columns={
#                     #     "model_name": "ModelName",
#                     #     "subtask": "SubtaskName",
#                     #     "index": "RowIndex",
#                     #     "prediction": "PredictedSentence",
#                     # })
#                     # #print(subtask_predictions_df.columns)
#                     # subtask_predictions_df.to_sql("BlimpSubtaskwiseBreakdown", if_exists="append", index=False, con=conn)
#                 except Exception as e:
#                     print(f"Error reading {score_file}")
#                     print(e)
#                     continue
#     else:
#         continue

# #subtasks_predictions_df = pd.concat(subtasks_predictions_df)
# #subtasks_predictions_df

In [16]:
# df1 = pd.DataFrame(test_subtask_list)

# #Use model and pivot to make subtask as columns
# df1 = df1.pivot(index="model_name", columns="subtask", values="score")
# df1

In [10]:
#Function block to read all predicted sentences for all models and store them in the database

# #Output Dump Folders Path
#  
# output_dump_folder_root = "/media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump"
# #subtasks_predictions_df = []

# for model_name in tqdm(os.listdir(output_dump_folder_root)):
#     if not os.path.isdir(os.path.join(output_dump_folder_root, model_name)):
#         continue
#     if "zeroshot" in os.listdir(os.path.join(output_dump_folder_root, model_name)):
#         model_output_dump_folder = os.path.join(output_dump_folder_root, model_name, "zeroshot")
#         if len(os.listdir(model_output_dump_folder)) == 17:
            
#             for subtask_name in os.listdir(model_output_dump_folder):
#                 predictions_file = os.path.join(model_output_dump_folder, subtask_name, "predictions.txt")

#                 try:
#                     subtask_predictions_df = pd.read_csv(predictions_file, sep="\t")
#                     subtask_predictions_df["model_name"] = model_name
#                     subtask_predictions_df["subtask"] = subtask_name
#                     #subtasks_predictions_df.append(subtask_predictions_df)
#                     subtask_predictions_df = subtask_predictions_df.rename(columns={
#                         "model_name": "ModelName",
#                         "subtask": "SubtaskName",
#                         "index": "RowIndex",
#                         "prediction": "PredictedSentence",
#                     })
#                     #print(subtask_predictions_df.columns)
#                     subtask_predictions_df.to_sql("BlimpSubtaskwiseBreakdown", if_exists="append", index=False, con=conn)
#                 except Exception as e:
#                     print(f"Error reading {predictions_file}")
#                     print(e)
#                     continue
#     else:
#         continue

# #subtasks_predictions_df = pd.concat(subtasks_predictions_df)
# #subtasks_predictions_df

  0%|          | 0/507 [00:00<?, ?it/s]